## **ANÁLISIS DE E-COMMERCE EN MÉXICO**

### *🗒️ DATOS Y GENERACIÓN*

---

#### **Objetivo:**
Generar un dataset realista de 10,000 registros de ecommerce mexicano con 39 variables y 5% de errores controlados para análisis de tiempos de entrega.

#### **Entregables:**
- Código de generación (`src/data_generation.py`)
- Dataset crudo con errores (`data/raw/dataset_raw.csv`)
- Dataset de muestra (`data/raw/dataset_sample_1000.csv`)

---
#### **INVESTIGACIÓN PRELIMINAR:**
Para asegurar el realismo de los datos, se realizó una investigación previa sobre el ecommerce mexicano:

- **Asociación Mexicana de Venta Online 2024:** Categorías más vendidas, métodos de pago

- **Transportistas** mexicanos: Estafeta, DHL, Correos de México, FedEx

- **Patrones estacionales:** Hot Sale (Mayo), Buen Fin (Noviembre), Navidad

- **Comportamiento del consumidor:** Preferencias por pagos, expectativas de entrega


#### **DATOS CLAVE ENCONTRADOS:**
- **Moda y Electrónicos** son las categorías más populares

- **Tarjeta de débito** es el método de pago más usado (48% de transacciones)

- **DHL** es más rápido pero caro, Correos es económico pero lento

- Solo ~50% de entregas llegan a tiempo en temporada alta




---

### **VARIABLES GENERADAS: (39 en total)**

#### 1. *Identificación* (2 variables)
| Variable | Descripción | Formato/Ejemplo |
|----------|-------------|-----------------|
| `order_id` | ID único del pedido | `ECOMMX-2024-00001` |
| `customer_id` | ID anónimo del cliente | `CUST-12345` |

#### 2. *Tiempos y Fechas* (8 variables - **LAS MÁS IMPORTANTES**)
| Variable | Descripción | Propósito del Análisis |
|----------|-------------|------------------------|
| `delivery_delay_days` | Días de retraso (real vs prometido) | **Variable objetivo principal** |
| `delivery_met_promise` | 1=Entregado a tiempo, 0=Retrasado | KPI de cumplimiento |
| `promised_delivery_days` | Días prometidos al cliente | Para medir expectativas |
| `actual_delivery_days` | Días reales de entrega | Para medir realidad |
| `order_date`, `shipped_date`, `delivered_date` | Fechas del proceso | Análisis temporal |
| `processing_days` | Días para procesar pedido | Eficiencia operativa |

#### 3. *Producto* (5 variables)
| Variable | Distribución | Rango Típico |
|----------|--------------|--------------|
| `product_category` | Moda (25%), Electrónicos (24%), Hogar (16%) | 7 categorías |
| `product_price_mxn` | Según categoría | $150 - $35,000 MXN |
| `product_weight_kg` | Según categoría | 0.1kg (moda) - 15kg (hogar) |
| `quantity` | 1-5 (95%), 6-10 (5%) | Reabastecimiento |
| `total_amount_mxn` | Precio × cantidad + envío | Ticket total |

#### 4. *Logística* (6 variables - **ENFOQUE DEL PROYECTO**)
| Variable | Opciones | Distribución |
|----------|----------|--------------|
| `shipping_carrier` | 7 transportistas mexicanos | Estafeta (28%), DHL (22%), Correos (15%) |
| `shipping_tier` | Estándar (85%), Express (15%) | Impacta costo y tiempo |
| `shipping_cost_mxn` | Según peso, distancia, carrier | $80 - $500 MXN |
| `distance_km` | Local, Regional, Nacional | 0-1,500 km |
| `shipping_type` | local, regional, nacional | Según destino |
| `shipping_cost_to_price_ratio` | costo_envío / precio_producto | Indicador de eficiencia |

#### 5. *Cliente* (6 variables)
| Variable | Categorías | Distribución |
|----------|------------|--------------|
| `customer_state` | 32 estados de México | Uniforme |
| `customer_region` | 5 regiones (Centro, Norte, etc.) | Agrupación para análisis |
| `customer_age_group` | 5 grupos etarios | 25-34 años (35%) |
| `customer_loyalty_months` | 0-60 meses | Media: 12 meses |
| `purchase_frequency` | Compras/mes | Media: 2.5 |
| `is_urban` | 1=Urbano (75%), 0=Rural | Cobertura logística |

#### 6. *Pago* (3 variables)
| Variable | Opciones | Distribución |
|----------|----------|--------------|
| `payment_method` | 5 métodos | Débito (48%), Crédito (30%), OXXO (15%) |
| `transaction_status` | 3 estados | Autorizada (67%), Rechazada, En revisión |
| `payment_installments` | 1-18 MSI | Solo para tarjeta de crédito |

#### 7.  *Canal* (2 variables)
| Variable | Opciones | Distribución |
|----------|----------|--------------|
| `sales_channel` | Marketplace (85%), D2C (15%) | Tipo de venta |
| `platform_name` | 6 plataformas | Mercado Libre (35%), Amazon (25%) |

#### 8. *Experiencia y Derivadas* (7 variables)
| Variable | Escala/Rango | Propósito |
|----------|--------------|-----------|
| `customer_delivery_rating` | 1-5 estrellas | Satisfacción del cliente |
| `delivery_issue` | 5 tipos de problemas | Análisis de "pain points" |
| `is_peak_season` | 0/1 | Hot Sale, Buen Fin, Navidad |
| `is_frequent_customer` | 0/1 | >3 compras/mes |
| `is_loyal_customer` | 0/1 | >12 meses de lealtad |
| `high_value_order` | 0/1 | >$5,000 MXN |
| `is_urban` | 0/1 | Zona urbana vs rural |


---

### **ESTRUCTURA TÉCNICA:**

#### *Generador:*

In [ ]:
class MexicoEcommerceGenerator:
    """
    Generador de datos con configuración específica para México
    Características:
    - Semilla 42 para reproducibilidad
    - Distribuciones basadas en investigación
    - Relaciones realistas entre variables
    - 5% de errores controlados
    """

In [ ]:
# %%
# Cargar librerías para mostrar el código
import pandas as pd
import numpy as np
from datetime import datetime

# Mostrar estructura básica del generador
print("ESTRUCTURA DEL GENERADOR - MexicoEcommerceGenerator")
print(" ")
print("""
MÉTODOS PRINCIPALES:
1. __init__(seed=42, n_records=10000) → Inicializa con semilla
2. _setup_mexico_data() → Configura datos reales de México
3. generate_dataset(include_errors=True) → Genera 10k registros
4. save_dataset() → Guarda en CSV

CLASES DE MÉTODOS:
• _generar_fechas_realistas() → Fechas con lógica de temporada
• _calcular_tiempo_entrega() → Tiempos por transportista
• _agregar_errores_controlados() → 5% de errores
• _generar_datos_producto() → Productos realistas por categoría
• _generar_datos_cliente() → Perfil demográfico mexicano
""")


---

### **RESULTADOS GENERADOS:**

In [4]:
# Cargar dataset generado para mostrar estadísticas
try:
    df = pd.read_csv('../data/raw/dataset_raw.csv', 
                     parse_dates=['order_date', 'shipped_date', 'delivered_date'])
    
    print("- ESTADÍSTICAS DEL DATASET GENERADO")
    print("" )
    print(f"Registros totales: {len(df):,}")
    print(f"Columnas: {len(df.columns)}")
    print(f"Período cubierto: {df['order_date'].min().date()} al {df['order_date'].max().date()}")
    print()
    
    # Estadísticas clave
    print("- MÉTRICAS DE ENTREGA (VARIABLES CLAVE):")
    print(f"• Entregas a tiempo: {df['delivery_met_promise'].mean():.1%}")
    print(f"• Días de retraso promedio: {df['delivery_delay_days'].mean():.1f}")
    print(f"• Calificación promedio: {df['customer_delivery_rating'].mean():.1f}/5.0")
    print()
    
    # Distribución de transportistas
    print("- DISTRIBUCIÓN DE TRANSPORTISTAS:")
    dist = df['shipping_carrier'].value_counts(normalize=True).head(7)
    for carrier, pct in dist.items():
        # Mostrar solo los principales (sin los typos)
        if not carrier.startswith(' '):
            print(f"  • {carrier:<20}: {pct:.1%}")
    
    print()
    print("- DISTRIBUCIÓN DE CATEGORÍAS:")
    cat_dist = df['product_category'].value_counts(normalize=True)
    for cat, pct in cat_dist.items():
        print(f"  • {cat:<20}: {pct:.1%}")
        
except FileNotFoundError:
    print("Dataset no encontrado. Ejecuta primero: python src/data_generation.py")
    print("\nEstadísticas de la última ejecución:")
    print("• Entregas a tiempo: 45.7%")
    print("• Días de retraso promedio: 1.1 días")
    print("• Calificación promedio: 4.1/5.0")
    print("• Transportista principal: Estafeta (27.8%)")

- ESTADÍSTICAS DEL DATASET GENERADO

Registros totales: 10,000
Columnas: 39
Período cubierto: 2024-12-25 al 2025-12-25

- MÉTRICAS DE ENTREGA (VARIABLES CLAVE):
• Entregas a tiempo: 45.7%
• Días de retraso promedio: 1.1
• Calificación promedio: 4.1/5.0

- DISTRIBUCIÓN DE TRANSPORTISTAS:
  • Estafeta            : 27.8%
  • DHL                 : 22.3%
  • FedEx               : 18.2%
  • Correos de México   : 14.5%
  • UPS                 : 9.9%
  • Redpack             : 4.8%
  • Paquetexpress       : 2.0%

- DISTRIBUCIÓN DE CATEGORÍAS:
  • Moda y Accesorios   : 25.4%
  • Electrónicos        : 23.8%
  • Hogar y Jardín      : 15.7%
  • Salud y Belleza     : 11.9%
  • Deportes            : 9.1%
  • Libros y Educación  : 8.0%
  • Juguetes y Bebés    : 6.1%



---

### Conclusión:

En esta primera etapa del proyecto se generaron los 10 000 registros con 39 variables basadas en una investigación del mercado, con `delivery_delay_days` y `delivery_met_promise` como nuestras variables claves. 

Se generaron el 5% de datos basura para su posterior limpieza. 